# 05.4 - Model Evaluation

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

How do we know if a model is good? This unit covers all the metrics for classification and regression, and how to choose the right one for your problem.

## 2. Why Does This Matter?

The metric you choose determines which model you pick. Accuracy can be misleading on imbalanced data. Understanding the confusion matrix, precision/recall, F1, ROC-AUC, and regression metrics is essential.

## 3. Prerequisites

- Phase 03 (Statistics), Units 05.2, 05.3

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Read a confusion matrix
- Compute and interpret precision, recall, F1, ROC-AUC
- Compute regression metrics (MAE, MSE, RMSE, R^2)
- Choose the right metric for a problem

## 5. Mental Model

Confusion matrix:

```
            Predicted
             0    1
Actual 0  [TN   FP]
       1  [FN   TP]
```

- Precision = TP / (TP + FP) - of predicted positives, how many are right
- Recall = TP / (TP + FN) - of actual positives, how many found
- F1 = harmonic mean of precision and recall
- ROC-AUC = probability a random positive ranks above a random negative


## 6. Generate Data and Train a Model

Use a synthetic classification dataset.


In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, n_redundant=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f"Test set: {len(X_test)} samples")


Test set: 300 samples


## 7. Confusion Matrix

The confusion matrix shows all four outcomes.


In [2]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"\nAccuracy = (TP+TN)/(total) = {(tp+tn)/(tp+tn+fp+fn):.3f}")


Confusion matrix:
[[135  23]
 [ 30 112]]

TN=135, FP=23, FN=30, TP=112

Accuracy = (TP+TN)/(total) = 0.823


## 8. Classification Metrics

Compute precision, recall, F1, and ROC-AUC.


In [3]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}")
print("\nFull report:")
print(classification_report(y_test, y_pred))


Accuracy:  0.823
Precision: 0.830
Recall:    0.789
F1:        0.809
ROC-AUC:   0.913

Full report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       158
           1       0.83      0.79      0.81       142

    accuracy                           0.82       300
   macro avg       0.82      0.82      0.82       300
weighted avg       0.82      0.82      0.82       300



## 9. Precision vs Recall Tradeoff

Precision and recall trade off against each other. The decision threshold controls this.


In [4]:
# Vary the decision threshold
for thresh in [0.3, 0.5, 0.7]:
    y_t = (y_proba > thresh).astype(int)
    p = precision_score(y_test, y_t)
    r = recall_score(y_test, y_t)
    print(f"Threshold {thresh}: precision={p:.3f}, recall={r:.3f}")
print("\nHigher threshold -> higher precision, lower recall.")
print("Lower threshold -> higher recall, lower precision.")


Threshold 0.3: precision=0.758, recall=0.972
Threshold 0.5: precision=0.830, recall=0.789
Threshold 0.7: precision=0.910, recall=0.570

Higher threshold -> higher precision, lower recall.
Lower threshold -> higher recall, lower precision.


## 10. Imbalanced Data: Why Accuracy Fails

On imbalanced data, accuracy is misleading.


In [5]:
# Create imbalanced data
X_imb, y_imb = make_classification(n_samples=1000, n_features=10, weights=[0.95, 0.05], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42)
m = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_p = m.predict(X_te)
print(f"Class balance: {y_te.mean():.2f}")
print(f"Accuracy:  {accuracy_score(y_te, y_p):.3f}")
print(f"Recall:    {recall_score(y_te, y_p):.3f}")
print(f"F1:        {f1_score(y_te, y_p):.3f}")
print("\nAccuracy looks high but recall is low - the model misses positives.")


Class balance: 0.07
Accuracy:  0.937
Recall:    0.150


F1:        0.240

Accuracy looks high but recall is low - the model misses positives.


## 11. Regression Metrics

For regression: MAE, MSE, RMSE, R^2.


In [6]:
from sklearn.linear_model import LinearRegression
np.random.seed(1)
Xr = np.random.uniform(0, 10, 300)
yr = 2 * Xr + 1 + np.random.normal(0, 1, 300)
mr = LinearRegression().fit(Xr.reshape(-1, 1), yr)
y_hat = mr.predict(Xr.reshape(-1, 1))

mae = mean_absolute_error(yr, y_hat)
mse = mean_squared_error(yr, y_hat)
rmse = np.sqrt(mse)
r2 = r2_score(yr, y_hat)
print(f"MAE:  {mae:.3f}")
print(f"MSE:  {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R^2:  {r2:.3f}")
print("\nMAE is in target units; RMSE penalizes large errors more.")


MAE:  0.807
MSE:  1.013
RMSE: 1.007
R^2:  0.972

MAE is in target units; RMSE penalizes large errors more.


## 12. Choosing the Right Metric

Match the metric to the business problem:
- **Spam detection**: high precision (don't block real email).
- **Cancer screening**: high recall (don't miss cancers).
- **Fraud detection**: F1 or ROC-AUC (imbalanced).
- **Pricing**: RMSE (penalize large errors).

## 13. Failure Case: Wrong Metric

Using accuracy on imbalanced data hides poor performance.


In [7]:
# Failure case: accuracy on imbalanced data
print("On the imbalanced dataset:")
print(f"  Accuracy = {accuracy_score(y_te, y_p):.3f} (looks good)")
print(f"  Recall   = {recall_score(y_te, y_p):.3f} (misses positives)")
print("\nA model predicting all 0s would get ~95% accuracy but recall=0.")


On the imbalanced dataset:
  Accuracy = 0.937 (looks good)
  Recall   = 0.150 (misses positives)

A model predicting all 0s would get ~95% accuracy but recall=0.


## 14. Debugging: Common Errors

- Using accuracy on imbalanced data.
- Using the wrong metric for the business goal.
- Evaluating on training data (overfitting).
- Not using probabilities for ranking metrics.

## 15. Real-World Considerations

- Always evaluate on a held-out test set.
- Use multiple metrics for a complete picture.
- Consider the cost of false positives vs false negatives.

## 16. Common Mistakes

- Reporting only accuracy.
- Tuning on the test set.
- Ignoring class imbalance.

## 17. When NOT to Use

- Don't use accuracy alone on imbalanced data.
- Don't use R^2 alone for nonlinear relationships.

## 18. Challenge

Compute the F1 score manually from the confusion matrix and verify it matches sklearn.


In [8]:
# Challenge: compute F1 manually
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1_manual = 2 * precision * recall / (precision + recall)
print(f"Manual F1: {f1_manual:.3f}")
print(f"sklearn F1: {f1_score(y_test, y_pred):.3f}")
print("\nThey match!")


Manual F1: 0.809
sklearn F1: 0.809

They match!


## Hands-On Practice

### Level 1: Basic
- Compute accuracy, precision, recall, F1 from a confusion matrix
- Plot an ROC curve and compute AUC

### Level 2: Guided
- Apply stratified k-fold cross-validation
- Compare 3 models on the same dataset using multiple metrics

### Level 3: Independent
- Build a model evaluation pipeline with preprocessing
- Analyze precision-recall tradeoff for a real problem

### Level 4: Realistic
- Implement nested cross-validation for unbiased performance estimation
- Create a comprehensive model comparison report

### Level 5: Challenge
- Build an automated model selection pipeline



## Knowledge Check

1. When should you use precision vs recall?
2. What is the advantage of cross-validation over a single train/test split?
3. What does an ROC AUC of 0.5 mean?



## Exit Criteria

- [ ] Can compute and interpret classification metrics
- [ ] Can use cross-validation properly
- [ ] Can compare models fairly
- [ ] Can select the right metric for the problem
- [ ] Completed Level 3+ practice



## Next Step

-> Unit 05.05: Decision Trees — interpretable, rule-based models.


## 19. Closed-Book Recall

Without looking back:

1. What does each cell of the confusion matrix mean?
2. Write precision and recall formulas.
3. When would you prefer recall over precision?
4. What is ROC-AUC?

## 20. Teach-Back Questions

Explain to another person:

- The precision-recall tradeoff.
- Why accuracy fails on imbalanced data.

## 21. Summary

You now understand how to evaluate models properly: confusion matrix, precision/recall/F1, ROC-AUC, and regression metrics. Choosing the right metric is critical.

## 22. Further Experiment

- Plot the precision-recall curve.
- Compute the F1 for different thresholds.

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
